In [1]:
import sympy as sp
from sympy import symbols, diff, solve

In [2]:
## Define Symbols
psi_dot, m, p, C_d, A, V_ref, V_y, F_x, e_vx, e_psi, e_y = symbols(
    "psi_dot m p C_d A V_ref V_y F_x e_vx e_psi e_y"
)
C_alp_f, C_alp_r, L_f, L_r, delta, I2, K = symbols("C_alp_f C_alp_r L_f L_r delta I2 K")

# State & Control Vectors
x = sp.Matrix([e_vx, V_y, psi_dot, e_y, e_psi])  # CTE is the last state
u = sp.Matrix([F_x, delta])  # Control inputs: longitudinal force and steering angle


# Logitudal Velocity Error Dynamics
f1 = psi_dot * V_y + 1 / m * (F_x - 0.5 * p * C_d * A * (V_ref + e_vx) ** 2)

# Lateral Velocity Error Dynamics
f2 = - psi_dot * (V_ref + e_vx) + 1 / m * (
    -C_alp_f * ((V_y + L_f * psi_dot) / (V_ref + e_vx) - delta) * sp.cos(delta)
    - C_alp_r * ((V_y - L_r * psi_dot) / (V_ref + e_vx))
)

# Yaw rate error dynamics
f3 = 1/ I2 * (L_f *
    -C_alp_f * ((V_y + L_f * psi_dot) / (V_ref + e_vx) - delta) * sp.cos(delta)
    - L_r * (- C_alp_r * ((V_y - L_r * psi_dot) / (V_ref + e_vx)))
)

# CTE Kinematics
f4 = (V_ref + e_vx) * sp.sin(e_psi) + V_y * sp.cos(e_psi)

# Heading Error Kinematics
f5 = psi_dot - K * (e_vx + V_ref)

# State Dynamics Vector
f = sp.Matrix([f1, f2, f3, f4, f5])

In [3]:
# Compute Jacobians
A = f.jacobian(x)
B = f.jacobian(u)
# Display Results
print("A Matrix (State Jacobian):")
sp.pprint(A)
print("\nB Matrix (Control Jacobian):")
sp.pprint(B)


A Matrix (State Jacobian):
⎡                  -0.5⋅A⋅C_d⋅p⋅(2⋅V_ref + 2⋅eᵥₓ)                              ↪
⎢                  ───────────────────────────────                             ↪
⎢                                 m                                            ↪
⎢                                                                              ↪
⎢         C_alp_f⋅(L_f⋅ψ_dot + V_y)⋅cos(δ)   Cₐₗₚ ᵣ⋅(-Lᵣ⋅ψ_dot + V_y)          ↪
⎢         ──────────────────────────────── + ────────────────────────      C_a ↪
⎢                               2                              2         - ─── ↪
⎢                  (V_ref + eᵥₓ)                  (V_ref + eᵥₓ)             V_ ↪
⎢-ψ_dot + ───────────────────────────────────────────────────────────    ───── ↪
⎢                                      m                                       ↪
⎢                                                                              ↪
⎢ C_alp_f⋅L_f⋅(L_f⋅ψ_dot + V_y)⋅cos(δ)   Cₐₗₚ ᵣ⋅Lᵣ⋅(-Lᵣ⋅ψ_dot + V_y)           ↪
⎢

In [4]:
# Evaluate at equilibrium point (e_vx=0, V_y=0, psi_dot=0, e_y=0, e_psi=0, F_x=0, delta=0)
equilibrium_point = {
    e_vx: 0,
    V_y: 0,
    psi_dot: 0,
    e_y: 0,
    e_psi: 0,
    F_x: 0,
    delta: 0,
}
A_eq = A.subs(equilibrium_point)
B_eq = B.subs(equilibrium_point)
print("\nA Matrix at Equilibrium:")
sp.pprint(A_eq)
print("\nB Matrix at Equilibrium:")
sp.pprint(B_eq)


A Matrix at Equilibrium:
⎡-1.0⋅A⋅C_d⋅V_ref⋅p                                                            ↪
⎢───────────────────              0                              0             ↪
⎢         m                                                                    ↪
⎢                                                                              ↪
⎢                          C_alp_f   Cₐₗₚ ᵣ                 C_alp_f⋅L_f   Cₐₗₚ ↪
⎢                        - ─────── - ──────               - ─────────── + ──── ↪
⎢                           V_ref    V_ref                     V_ref        V_ ↪
⎢         0              ──────────────────      -V_ref + ──────────────────── ↪
⎢                                m                                    m        ↪
⎢                                                                              ↪
⎢                                                                2             ↪
⎢                       C_alp_f⋅L_f   Cₐₗₚ ᵣ⋅Lᵣ       C_alp_f⋅L_f    Cₐₗₚ ᵣ⋅Lᵣ ↪
⎢ 